In [ ]:
# ===========================================================
# 1. IMPORTS
# ===========================================================
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
sns.set()
import warnings
warnings.filterwarnings("ignore")

from azureml.core import Workspace, Dataset

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from statsmodels.stats.outliers_influence import variance_inflation_factor


# ===========================================================
# 2. LOAD DATASET FROM AZURE ML ASSET
# ===========================================================
subscription_id = 'bbeaebad-8e6b-4196-95cc-514eaacabb7c'
resource_group = 'CMR'
workspace_name = 'CMRMLOps'

workspace = Workspace(subscription_id, resource_group, workspace_name)

dataset = Dataset.get_by_name(workspace, name='data')   # correct name
df = dataset.to_pandas_dataframe()

print("Dataset loaded successfully\n")
print(df.head())


# ===========================================================
# 3. DATA UNDERSTANDING
# ===========================================================
print("\n--- Basic Info ---")
print(df.info())

print("\n--- Statistical Summary ---")
print(df.describe())

target = "House_Price"
features = [c for c in df.columns if c != target]


# ===========================================================
# 4. EXPLORATORY DATA ANALYSIS
# ===========================================================

# Distribution of target
plt.figure(figsize=(6,4))
sns.histplot(df[target], kde=True)
plt.title("Distribution of House Prices")
plt.show()

# Area vs Price
plt.figure(figsize=(6,4))
sns.scatterplot(x=df["Area_sqft"], y=df[target])
plt.title("Area vs Price")
plt.show()

# Age vs Price
plt.figure(figsize=(6,4))
sns.scatterplot(x=df["Age_of_House"], y=df[target])
plt.title("Age of House vs Price")
plt.show()

# Correlation
plt.figure(figsize=(10,6))
sns.heatmap(df.corr(), annot=True, cmap="coolwarm")
plt.title("Correlation Heatmap")
plt.show()


# ===========================================================
# 5. SIMPLE LINEAR REGRESSION (Area → Price)
# ===========================================================
X_simple = df[["Area_sqft"]]
y = df[target]

model_simple = LinearRegression()
model_simple.fit(X_simple, y)

print("\nSimple Regression Slope:", model_simple.coef_[0])
print("Simple Regression Intercept:", model_simple.intercept_)

# Plot
plt.scatter(X_simple, y)
plt.plot(X_simple, model_simple.predict(X_simple), color='red')
plt.title("Simple Linear Regression")
plt.show()


# ===========================================================
# 6. MULTIPLE LINEAR REGRESSION
# ===========================================================
X = df[["Area_sqft", "Bedrooms", "Bathrooms", "Floors", "Age_of_House"]]
y = df[target]

model_multi = LinearRegression()
model_multi.fit(X, y)

print("\n--- Multiple Regression Coefficients ---")
for col, coef in zip(X.columns, model_multi.coef_):
    print(f"{col}: {coef}")

importance = pd.DataFrame({"Feature": X.columns, "Coefficient": model_multi.coef_})
print("\nFeature Importance:\n", importance)


# ===========================================================
# 7. MULTICOLLINEARITY (VIF)
# ===========================================================
vif_df = pd.DataFrame()
vif_df["Feature"] = X.columns
vif_df["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
print("\n--- VIF Table ---")
print(vif_df)


# ===========================================================
# 8. OUTLIER DETECTION
# ===========================================================
plt.figure(figsize=(6,4))
sns.boxplot(df[target])
plt.title("Price Outliers")
plt.show()

df["z_price"] = (df[target] - df[target].mean()) / df[target].std()
outliers = df[df["z_price"].abs() > 3]

print("\nOutliers detected:")
print(outliers[[target, "z_price"]])


# ===========================================================
# 9. TRAIN–TEST SPLIT + MODEL EVALUATION
# ===========================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model_multi.fit(X_train, y_train)
y_pred = model_multi.predict(X_test)

r2 = r2_score(y_test, y_pred)
adj_r2 = 1 - ((1 - r2) * (len(y)-1) / (len(y) - X.shape[1] - 1))

print("\nR²:", r2)
print("Adjusted R²:", adj_r2)
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))


# ===========================================================
# 10. SCALING
# ===========================================================
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

scaled_model = LinearRegression()
scaled_model.fit(X_scaled, y)

print("\nScaled Model R²:", scaled_model.score(X_scaled, y))


# ===========================================================
# 11. RESIDUAL ANALYSIS
# ===========================================================
residuals = y_test - y_pred

plt.scatter(y_pred, residuals)
plt.axhline(0, color="red")
plt.title("Residuals vs Predictions")
plt.show()

sns.histplot(residuals, kde=True)
plt.title("Residual Distribution")
plt.show()


# ===========================================================
# 12. POLYNOMIAL REGRESSION (Area²)
# ===========================================================
poly = PolynomialFeatures(degree=2)
X_poly = poly.fit_transform(df[["Area_sqft"]])

poly_model = LinearRegression()
poly_model.fit(X_poly, y)

print("\nPolynomial Regression R²:", poly_model.score(X_poly, y))


# ===========================================================
# 13. LOCATION-ONLY MODEL
# ===========================================================
loc_features = ["Distance_to_City_km", "Nearby_Schools", "Crime_Rate"]
X_loc = df[loc_features]

loc_model = LinearRegression()
loc_model.fit(X_loc, y)

print("\nLocation-only Model R²:", loc_model.score(X_loc, y))


# ===========================================================
# 14. REDUCED MODEL (Area + Bedrooms + Bathrooms)
# ===========================================================
X_reduced = df[["Area_sqft", "Bedrooms", "Bathrooms"]]
reduced_model = LinearRegression()
reduced_model.fit(X_reduced, y)

print("\nReduced Model R²:", reduced_model.score(X_reduced, y))


# ===========================================================
# 15. PREDICT NEW HOUSE
# ===========================================================
new_house = pd.DataFrame({
    "Area_sqft":[1650],
    "Bedrooms":[3],
    "Bathrooms":[2],
    "Floors":[2],
    "Age_of_House":[5]
})

pred_price = model_multi.predict(new_house)[0]
print("\nPredicted Price for New House:", pred_price)


# ===========================================================
# 16. BUSINESS INSIGHTS
# ===========================================================
print("\n--- BUSINESS INTERPRETATION ---")
print("• Larger area strongly increases price.")
print("• More bedrooms and bathrooms → higher price.")
print("• Older homes (Age_of_House↑) tend to have lower prices.")
print("• Homes closer to city are priced higher.")
print("• Higher crime rate reduces price.")
print("• Nearby schools increase attractiveness.")